<a href="https://colab.research.google.com/github/dan-zeman/belo-horizonte/blob/main/Udapi3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universal Dependencies with `udapi-python`

This is the third session with the `udapi-python` library. As we start with a blank virtual machine, we must start again with installing Udapi and downloading the treebank data.

## 1. Install `udapi`

First, we need to install the `udapi` Python library. This can be done using `pip`.

In [ ]:
%%bash
# (Preceding a single line with ! makes that line interpreted by shell instead of python.
# Inserting %%bash at the beginning of the cell makes the whole cell interpreted by bash.)
pip install --upgrade udapi
udapy -h

## 2. Download a UD Treebank

Download one or more UD treebanks from GitHub. Feel free to add downloads of other treebanks you are interested in. See https://universaldependencies.org/ for the list of available treebanks. Besides the UD homepage, you can also try https://universaldependencies.org/languages.html, where you can filter languages by family and genus.

Smart tip: If you want to work with your own data instead, click on the Files icon in the left pane of Colab, create a treebank folder on the local disk and upload your CoNLL-U file(s) there. Then you will be able to say that this is the treebank you want to load and process.

In [ ]:
%%bash
rm -rf UD_*
git clone https://github.com/UniversalDependencies/UD_English-GUM.git

## 3. Load the Treebank in Python

Now we will access the treebank data from Python. The `udapi.core.document.Document` class is used to load the treebank. You can edit the code cell below and specify the treebank you want to work with (it must be one of the treebanks you downloaded above).

**Important:** In some of the cells below, we modify the data (mark nodes) inside Python's memory. If we want to modify and run those cells again, we may need to reload a clean version of the treebank from the disk first. To achieve that, we will have to go back to this section and run this cell again.

In [ ]:
import glob
from udapi.core.document import Document

# Read the CoNLL-U files.
###############################################################################
# TODO: Change this variable to explore different datasets. It must be one of
# those you downloaded above, or saved manually through Colab's Files icon.
treebank = 'UD_English-GUM'

# List all .conllu files in the specified treebank folder.
conllu_files = glob.glob(f"{treebank}/*.conllu")
print(f"Found {len(conllu_files)} CoNLL-U files in {treebank}:")
for file in sorted(conllu_files):
    print(file)
print(f"Loading {treebank}...")
# Each CoNLL-U file will be stored as one Document object.
# Note that some treebanks use the '# newdoc' comment to mark document boundaries within each file.
# That is a different notion of document!
filedocs = []
nsent = 0
for file in sorted(conllu_files):
    doc = Document(filename=file)
    filedocs.append(doc)
    nsent += len(doc.bundles)
print(f"Loaded {nsent} sentences from {treebank}.")



# The following code does not belong to loading a treebank but it is also
# initialization of something that we will need later.

from collections import defaultdict
import pandas as pd

def generate_two_dimensional_table(clusters, display_mode='#'):
    """
    Generates, sorts, and displays a two-dimensional table from a clusters dictionary.

    Args:
        clusters (dict): A dictionary where keys are (parent_tag, child_tag) tuples
                         and values are counts.
        display_mode (str): The mode for displaying cell values.
                            Options: '#': absolute counts,
                                     'H%': horizontal percentage (row-wise),
                                     'V%': vertical percentage (column-wise),
                                     '%': percentage of grand total.
    Returns:
        pandas.DataFrame: The DataFrame used for display.
    """
    import pandas as pd

    # Get all unique parent and child tags
    parent_tags = sorted(list(set([pair[0] for pair in clusters.keys()])))
    child_tags = sorted(list(set([pair[1] for pair in clusters.keys()])))

    # Create a DataFrame to store the counts (initial data only)
    df_raw_counts = pd.DataFrame(0, index=parent_tags, columns=child_tags)

    # Populate the DataFrame with counts from the clusters dictionary
    for (parent, child), count in clusters.items():
        df_raw_counts.loc[parent, child] = count

    # Calculate row sums for sorting purposes
    row_sums_for_sorting = df_raw_counts.sum(axis=1)

    # Sort rows based on these sums in descending order
    sorted_row_indices = row_sums_for_sorting.sort_values(ascending=False).index
    df_raw_counts = df_raw_counts.reindex(sorted_row_indices)

    # Calculate column sums for sorting purposes (after row sorting)
    column_sums_for_sorting = df_raw_counts.sum(axis=0)

    # Sort columns based on these sums in descending order
    sorted_column_names = column_sums_for_sorting.sort_values(ascending=False).index
    df_raw_counts = df_raw_counts.reindex(columns=sorted_column_names)

    # --- Prepare DataFrame for display based on display_mode ---
    df_display = df_raw_counts.copy() # Start with raw counts

    # Calculate absolute totals for later use (these will always be absolute counts)
    absolute_row_totals = df_raw_counts.sum(axis=1)
    absolute_column_totals = df_raw_counts.sum(axis=0)
    absolute_grand_total = df_raw_counts.sum().sum()

    if display_mode == 'H%':
        # Horizontal percentage: percent of the total count of the row
        df_display = df_raw_counts.div(absolute_row_totals.replace(0, 1), axis=0) * 100
        df_display = df_display.round(2).fillna(0)
        df_display = df_display.map(lambda x: f'{x:.2f}%')
    elif display_mode == 'V%':
        # Vertical percentage: percent of the total count of the column
        df_display = df_raw_counts.div(absolute_column_totals.replace(0, 1), axis=1) * 100
        df_display = df_display.round(2).fillna(0)
        df_display = df_display.map(lambda x: f'{x:.2f}%')
    elif display_mode == '%':
        # Percent of the total count in the table
        df_display = (df_raw_counts / absolute_grand_total) * 100
        df_display = df_display.round(2).fillna(0)
        df_display = df_display.map(lambda x: f'{x:.2f}%')
    elif display_mode == '#':
        # Absolute counts (already in df_raw_counts, convert to string for display consistency)
        df_display = df_raw_counts.astype(str)
    else:
        raise ValueError("Invalid display_mode. Choose from '#', 'H%', 'V%', '%'.")

    # Add the absolute Row_Total column
    df_display['Total'] = absolute_row_totals.astype(str)

    # Add the absolute Col_Total row
    col_total_row_for_display = absolute_column_totals.astype(str)
    col_total_row_for_display['Total'] = str(absolute_grand_total) # Grand total
    df_display.loc['Total'] = col_total_row_for_display

    return df_display

## 4. Tree Examples for Papers in LaTeX (Overleaf)

This is a continuation of Session 2, where we learned how to display selected trees directly in Colab, and how to generate a HTML document with highlighted search results, which can then be downloaded from Colab.

If you are writing an article about syntax and need tree diagrams, Udapi can help you generate them from the treebank. It uses the LaTeX publishing system (you may know it as the Overleaf web service). If you are writing your paper in LaTeX, you may download the source code of the examples and insert it in the source code of your paper. If you are writing it in anything else, you may run LaTeX in Colab, it will generate a PDF with the trees and you can then insert them to your document as vector graphics.

### Install a LaTeX compiler

First, we need to ensure a LaTeX compiler is available. Colab environments don't always come with a full TeX distribution, so we'll install a minimal one using `apt-get`.

In [ ]:
%%bash
apt-get update
apt-get install -y texlive-latex-base texlive-fonts-recommended texlive-latex-extra texlive-xetex

### Use Udapi to select examples and save them as LaTeX

Udapi contains a block `write.Tikz`, which generates code understandable by LaTeX. It could generate such code for all trees in the treebank but that is probably much more than you need, so we will also instruct Udapi to filter the treebank and write only some trees. Examples in papers often need to be short so that they fit on the page, so the example code here demonstrates how to select trees that contain exactly 6 nodes. Naturally, you will also want to add criteria that characterize the examples you intend to show in the paper.

In [ ]:
%%bash
# Create examples.tex, the source code for LaTeX.
# The util.Filter block of Udapi is instructed to keep only trees of exactly 6 nodes.
cat UD_English-GUM/en_gum-ud-test.conllu \
  | udapy util.Filter keep_tree='len(tree.descendants) == 6' \
          write.Tikz > examples.tex
# Compile the source code with LaTeX. This will create examples.pdf.
# Then you can download:
# - examples.tex and insert it to the source code of your paper (Overleaf)
# - examples.pdf and insert it to a document as vector graphic
xelatex examples.tex

### Download the files

Finally, you can download the files (PDF and its LaTeX source).

In [21]:
from google.colab import files

files.download('examples.pdf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download('examples.tex')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Enhanced Dependencies in Udapi

Udapi has some support for processing enhanced dependencies but it cannot visualize them.

In [ ]:
clusters = defaultdict(int)

for doc in filedocs:
    # Bundle is a unit that corresponds to one sentence in Udapi.
    # In theory it may have multiple trees for that sentence; but normally there is just one.
    for bundle in doc.bundles:
        root = bundle.get_tree()
        # When we were working with basic dependency trees, root.descendants was
        # enough. But with enhanced dependency graphs, we need also empty nodes.
        nodes = root.descendants_and_empty
        for node in nodes:
            ###################################################################
            # TODO: You can modify your search criteria here.
            # Each node can have multiple incoming dependencies (and parents).
            # In CoNLL-U, they are stored in the ninth column (called DEPS).
            # node.deps returns a list of dictionary structures, with the
            # elements indexed 'parent' and 'deprel'. The first one is a node
            # object, the second one is a string.
            edeprels = [dep['deprel'] for dep in node.deps]
            for edeprel in edeprels:
                if edeprel.startswith('obl'):
                    ############################################################
                    # TODO: You can modify your clustering keys here (rows and
                    # columns of the resulting table).
                    pair = (edeprel, node.form.lower())
                    clusters[pair] += 1

# Now display the clusters as a table.

###############################################################################
# TODO: Set your preferred display mode here.
# Options: '#':  absolute counts
#          'H%': horizontal percentage (percent of the total count of the row)
#          'V%': vertical percentage (percent of the total count of the column)
#          '%':  percent of the total count in the table
display_mode = '#' # Change this variable to select the desired display mode

# Generate the table of clusters.
df_display = generate_two_dimensional_table(clusters, display_mode=display_mode)

# Display the table
print(f"Enhanced oblique deprel (rows) vs. lowercased child form (columns) (Mode: {display_mode}):")
display(df_display)


In [ ]:
# @title Download the table for further analysis
from google.colab import files

# Define the filename for the CSV file
output_csv_filename = 'clusters.tsv'

# Save the DataFrame to a tab-separated CSV file
df_display.to_csv(output_csv_filename, sep='\t', encoding='utf-8')

print(f"File '{output_csv_filename}' is ready for download.")
files.download(output_csv_filename)